In [ ]:
from pathlib import Path
import os

import open3d as o3d
import numpy as np
from tqdm import tqdm
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Проверить доступность CUDA в Open3D
print("CUDA available:", o3d.core.cuda.is_available())

In [ ]:
DATA_ROOT = Path("dataset_noisy")
SPLITS = ["train", "val", "test"]
CLASSES = ["sphere", "cube", "cylinder", "cone", "torus"]
CLS_TO_IDX = {c: i for i, c in enumerate(CLASSES)}

# Оптимизированные параметры препроцессинга
VOXEL_SIZE = 0.01  # Увеличил для лучшей обработки шума
RADIUS_NORMAL = 4.0 * VOXEL_SIZE
MAX_NN_NORMAL = 30  # Уменьшил для стабильности

# FPFH параметры
RADIUS_FPFH = 6.0 * VOXEL_SIZE
MAX_NN_FPFH = 100

# ISS keypoints (опционально)
ISS_SALIENT_RADIUS = 8.0 * VOXEL_SIZE
ISS_NON_MAX_RADIUS = 6.0 * VOXEL_SIZE
ISS_GAMMA_21 = 0.9
ISS_GAMMA_32 = 0.9
ISS_MIN_NEIGHBORS = 3

def load_point_cloud(path: Path) -> o3d.geometry.PointCloud:
    pcd = o3d.io.read_point_cloud(str(path))
    return pcd

def robust_preprocess(pcd: o3d.geometry.PointCloud,
                     voxel_size: float = VOXEL_SIZE,
                     radius_normal: float = RADIUS_NORMAL,
                     max_nn_normal: int = MAX_NN_NORMAL) -> o3d.geometry.PointCloud:
    """Улучшенный препроцессинг с обработкой вырожденных случаев"""
    
    # Фильтрация выбросов
    pcd, ind = pcd.remove_statistical_outlier(nb_neighbors=20, std_ratio=2.0)
    
    if voxel_size and len(pcd.points) > 100:  # Проверка на минимальное количество точек
        pcd = pcd.voxel_down_sample(voxel_size)
    
    # Проверка что осталось достаточно точек
    if len(pcd.points) < 10:
        raise ValueError(f"Слишком мало точек после даунсэмплинга: {len(pcd.points)}")
        
    try:
        pcd.estimate_normals(
            o3d.geometry.KDTreeSearchParamHybrid(radius=radius_normal, max_nn=max_nn_normal)
        )
        pcd.orient_normals_consistent_tangent_plane(min(max_nn_normal, len(pcd.points)))
    except Exception as e:
        print(f"Warning: нормали не вычислены: {e}")
        # Создаем искусственные нормали если не удалось вычислить
        pcd.normals = o3d.utility.Vector3dVector(np.random.rand(len(pcd.points), 3))
    
    return pcd

def compute_enhanced_fpfh(pcd: o3d.geometry.PointCloud,
                         radius_feature: float = RADIUS_FPFH,
                         max_nn: int = MAX_NN_FPFH) -> np.ndarray:
    """Улучшенный FPFH с дополнительными статистиками"""
    try:
        fpfh = o3d.pipelines.registration.compute_fpfh_feature(
            pcd,
            o3d.geometry.KDTreeSearchParamHybrid(radius=radius_feature, max_nn=min(max_nn, len(pcd.points)))
        ).data
        
        if fpfh.shape[1] < 3:  # Минимальное количество точек для статистик
            # Возвращаем нулевой вектор правильной размерности
            return np.zeros(99, dtype=np.float32)
            
        # Базовые статистики
        mu = fpfh.mean(axis=1)
        sd = fpfh.std(axis=1)
        median = np.median(fpfh, axis=1)
        
        # Дополнительные статистики
        q25 = np.percentile(fpfh, 25, axis=1)
        q75 = np.percentile(fpfh, 75, axis=1)
        skew = ((fpfh - mu[:, np.newaxis]) ** 3).mean(axis=1) / (sd ** 3 + 1e-8)
        
        # Объединяем все признаки
        enhanced_features = np.concatenate([mu, sd, median, q25, q75, skew])
        return enhanced_features.astype(np.float32)
        
    except Exception as e:
        print(f"Warning: FPFH computation failed: {e}")
        return np.zeros(99, dtype=np.float32)  # 33 * 3 = 99

def compute_iss_keypoints(pcd: o3d.geometry.PointCloud) -> o3d.geometry.PointCloud:
    """Вычисление ISS ключевых точек с обработкой ошибок"""
    try:
        if len(pcd.points) < ISS_MIN_NEIGHBORS * 2:
            return pcd  # Возвращаем исходные точки если слишком мало
            
        kp = o3d.geometry.keypoint.compute_iss_keypoints(
            pcd,
            salient_radius=ISS_SALIENT_RADIUS,
            non_max_radius=ISS_NON_MAX_RADIUS,
            gamma_21=ISS_GAMMA_21,
            gamma_32=ISS_GAMMA_32,
            min_neighbors=ISS_MIN_NEIGHBORS
        )
        
        # Если ключевых точек слишком мало, используем равномерную выборку
        if len(kp.points) < 10:
            kp = pcd.farthest_point_down_sample(min(100, len(pcd.points)))
            
        return kp
    except Exception as e:
        print(f"Warning: ISS failed: {e}")
        return pcd.farthest_point_down_sample(min(100, len(pcd.points)))

def compute_global_features(pcd: o3d.geometry.PointCloud) -> np.ndarray:
    """Вычисление глобальных геометрических признаков"""
    points = np.asarray(pcd.points)
    
    if len(points) < 10:
        return np.zeros(15, dtype=np.float32)
    
    # Статистики распределения точек
    centroid = points.mean(axis=0)
    cov = np.cov(points.T)
    eigenvalues = np.linalg.eigvalsh(cov)
    eigenvalues.sort()
    
    # Геометрические признаки
    bounding_box = pcd.get_axis_aligned_bounding_box()
    bbox_size = bounding_box.get_extent()
    bbox_volume = bbox_size[0] * bbox_size[1] * bbox_size[2]
    
    # Вычисляем плотность
    volume = max(bbox_volume, 1e-8)
    density = len(points) / volume
    
    # Собираем все глобальные признаки
    global_features = np.concatenate([
        eigenvalues / (eigenvalues.sum() + 1e-8),  # Нормализованные собственные значения (3)
        bbox_size / (bbox_size.max() + 1e-8),      # Нормализованный размер bbox (3)
        [density / 1000.0],                        # Нормализованная плотность (1)
        centroid / (np.linalg.norm(centroid) + 1e-8)  # Нормализованный центроид (3)
    ])
    
    return global_features.astype(np.float32)

def extract_comprehensive_features(path: Path) -> np.ndarray:
    """Извлечение комплексных признаков: FPFH + глобальные признаки"""
    pcd = load_point_cloud(path)
    pcd = robust_preprocess(pcd)
    
    # 1. FPFH на всех точках
    fpfh_features = compute_enhanced_fpfh(pcd)
    
    # 2. Глобальные геометрические признаки
    global_features = compute_global_features(pcd)
    
    # 3. FPFH на ключевых точках (опционально)
    keypoints = compute_iss_keypoints(pcd)
    if len(keypoints.points) > 10:
        kp_fpfh = compute_enhanced_fpfh(keypoints)
    else:
        kp_fpfh = np.zeros_like(fpfh_features)
    
    # Объединяем все признаки
    all_features = np.concatenate([fpfh_features, global_features, kp_fpfh])
    return all_features

def iter_split_files(root: Path, split: str):
    base = root / split
    for cls in CLASSES:
        cls_dir = base / cls
        if not cls_dir.exists():
            continue
        for fn in os.listdir(cls_dir):
            if fn.lower().endswith((".ply", ".pcd", ".xyz")):
                yield cls, (cls_dir / fn)

def build_split_matrix(root: Path, split: str):
    X, y, ids = [], [], []
    for cls, path in tqdm(list(iter_split_files(root, split)),
                          desc=f"{split}", unit="obj"):
        try:
            vec = extract_comprehensive_features(path)
            X.append(vec)
            y.append(CLS_TO_IDX[cls])
            ids.append(str(path))
        except Exception as e:
            print(f"[WARN] {path}: {e}")
    
    if len(X) == 0:
        return np.empty((0,)), np.empty((0,)), []
    
    X = np.stack(X)
    y = np.array(y, dtype=np.int64)
    return X, y, ids

def train_and_eval(Xtr, ytr, Xval, yval, Xte, yte, model, label: str):
    print(f"\n{'='*60}")
    print(f"Training {label}")
    print(f"{'='*60}")
    
    model.fit(Xtr, ytr)
    
    for split_name, Xs, ys in [("val", Xval, yval), ("test", Xte, yte)]:
        yp = model.predict(Xs)
        acc = accuracy_score(ys, yp)
        print(f"[{label}] {split_name} accuracy: {acc:.4f}")
        print(classification_report(ys, yp, target_names=CLASSES, digits=4))
        print("Confusion Matrix:")
        print(confusion_matrix(ys, yp))
        print("-"*60)


In [ ]:
# Основной пайплайн
print("Building feature matrices...")
feature_sets = {}
for split in SPLITS:
    X, y, ids = build_split_matrix(DATA_ROOT, split)
    feature_sets[split] = {"X": X, "y": y, "ids": ids}
    print(f"{split}: {X.shape}, class distribution: {np.bincount(y, minlength=len(CLASSES))}")

Xtr = feature_sets["train"]["X"]
ytr = feature_sets["train"]["y"]
Xval = feature_sets["val"]["X"]
yval = feature_sets["val"]["y"]
Xte  = feature_sets["test"]["X"]
yte  = feature_sets["test"]["y"]

# Проверяем что данные не пустые
if len(Xtr) == 0 or len(Xval) == 0 or len(Xte) == 0:
    raise ValueError("Один из сплитов пустой! Проверьте пути к данным.")

print(f"\nFeature dimensions: {Xtr.shape[1]}")

# Тестируем несколько моделей
models = {
    "KNN": make_pipeline(
        StandardScaler(),
        KNeighborsClassifier(n_neighbors=5, metric="euclidean", weights="distance")
    ),
    "SVM-RBF": make_pipeline(
        StandardScaler(),
        SVC(kernel="rbf", C=1.0, gamma="scale", class_weight='balanced')
    ),
    "SVM-Linear": make_pipeline(
        StandardScaler(),
        SVC(kernel="linear", C=0.1, class_weight='balanced')
    ),
    "RandomForest": make_pipeline(
        StandardScaler(),
        RandomForestClassifier(n_estimators=100, max_depth=20, 
                             class_weight='balanced', random_state=42)
    )
}

for name, model in models.items():
    train_and_eval(Xtr, ytr, Xval, yval, Xte, yte, model, name)

# Анализ важности признаков (для RandomForest)
if "RandomForest" in models:
    rf_model = models["RandomForest"].steps[1][1]
    print("\nFeature importance analysis:")
    print(f"Total features: {len(rf_model.feature_importances_)}")
    print(f"Top 10 feature importances: {np.sort(rf_model.feature_importances_)[-10:]}")